# Fairness Analysis — Proxy-Group Audit

## Runner Injury Risk Prediction — Lovdal et al. (2021) Replication

This notebook performs a **fairness audit** of the tuned XGBoost models from
notebooks 04 and 05. Since the dataset contains no demographic attributes
(age, sex, ethnicity are masked), we construct **proxy groups** based on
observable training characteristics and evaluate whether the model performs
equitably across these groups.

### Contents

1. Data loading — models + test sets for day and week approaches
2. Proxy group creation — volume, injury history, data density
3. Volume-based fairness analysis
4. Injury-history fairness analysis
5. Data-density fairness analysis
6. Cross-method summary — heatmaps
7. Day vs week comparison
8. Critical discussion — limits without demographics
9. Summary

### Key concepts

- **Proxy groups**: in the absence of protected attributes, we split athletes
  by training volume (high/low), injury history (ever/never injured), and data
  density (many/few observations). These proxies can reveal systematic
  performance gaps even without demographic data.
- **Disparity ratio**: for each metric, `group_value / reference_value`.
  A ratio of 1.0 means parity. The **four-fifths rule** (EEOC guideline,
  adapted) flags ratios below 0.8 as potential adverse impact.
- **Limitations**: proxy groups are not protected attributes — they cannot
  replace a demographic fairness audit, but they surface performance gaps
  that would otherwise go undetected.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


# Ensure src/ is importable regardless of the notebook working directory
def _find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "src").exists() or (candidate / "pyproject.toml").exists():
            return candidate
    return start

PROJECT_ROOT = _find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import INJURY_COL, RANDOM_SEED
from src.data_loading import get_feature_columns
from src.fairness.audit import (
    compute_disparity_ratios,
    compute_group_metrics,
    create_athlete_groups,
    plot_disparity_ratios,
    plot_fairness_summary_heatmap,
    plot_group_metrics_bars,
)
from src.preprocessing.io import load_model, load_splits
from src.utils.logging_config import setup_logging
from src.utils.plotting import save_figure, set_style
from src.utils.reproducibility import set_global_seed

# --- Reproducibility & style ---
set_global_seed(RANDOM_SEED)
setup_logging()
set_style()

---

## 1. Data Loading

We load the tuned XGBoost models and preprocessed test sets, then generate
predictions (binary + probabilities) needed for per-group metric computation.

In [ ]:
# --- Day approach ---
day_model = load_model("day_best_model")
day_train, day_test = load_splits(prefix="day")
feature_cols_day = get_feature_columns(day_test)
X_test_day = day_test[feature_cols_day]
y_test_day = day_test[INJURY_COL]

y_prob_day = day_model.predict_proba(X_test_day)[:, 1]
y_pred_day = (y_prob_day >= 0.5).astype(int)

# --- Week approach ---
week_model = load_model("week_best_model")
week_train, week_test = load_splits(prefix="week")
feature_cols_week = get_feature_columns(week_test)
X_test_week = week_test[feature_cols_week]
y_test_week = week_test[INJURY_COL]

y_prob_week = week_model.predict_proba(X_test_week)[:, 1]
y_pred_week = (y_prob_week >= 0.5).astype(int)

print("Day approach:")
print(f"  Test set: {X_test_day.shape[0]:,} rows x {X_test_day.shape[1]} features")
print(f"  Injury rate: {y_test_day.mean():.2%}")
print(f"  Predicted positives: {y_pred_day.sum()} ({y_pred_day.mean():.2%})")

print("\nWeek approach:")
print(f"  Test set: {X_test_week.shape[0]:,} rows x {X_test_week.shape[1]} features")
print(f"  Injury rate: {y_test_week.mean():.2%}")
print(f"  Predicted positives: {y_pred_week.sum()} ({y_pred_week.mean():.2%})")

---

## 2. Proxy Group Creation

We construct three proxy grouping strategies for each approach:

| Method | Groups | Rationale |
|--------|--------|-----------|
| **Volume** | high / low (median split on total km) | High-volume athletes may have more training signal |
| **Injury history** | ever / never injured (in test set) | Athletes with prior injuries may be systematically different |
| **Data density** | high / low (median split on observation count) | Athletes with more data may get better predictions |

**Note on `reference_df`**: because the train/test split is athlete-disjoint
(`GroupShuffleSplit`), training-set athlete summaries cannot be mapped onto the
test set. Volume and data density are computed directly from the test set
(they use features and row counts, not the target — no leakage). Injury history
uses `INJURY_COL` from the test set; this is standard for post-hoc fairness
evaluation (analogous to stratifying recall by class) but introduces a mild
circularity that should be kept in mind when interpreting results.

In [ ]:
# --- Day approach: create proxy groups ---
# Volume and data_density are computed directly from the test set — they use
# feature columns and row counts (not the target), so there is no leakage.
# Injury_history uses INJURY_COL, but this is acceptable for post-hoc fairness
# evaluation: grouping by outcome to measure per-group performance is standard
# practice (analogous to stratifying recall/precision by class).  Note that
# with athlete-disjoint splits, reference_df=train cannot be used because
# train athletes are absent from the test set.
day_groups_volume = create_athlete_groups(
    day_test, method="volume", feature_col="day_0_total_km",
)
day_groups_injury = create_athlete_groups(
    day_test, method="injury_history",
)
day_groups_density = create_athlete_groups(
    day_test, method="data_density",
)

print("Day approach — proxy group distributions:")
print(f"\n  Volume:\n{day_groups_volume.value_counts().to_string()}")
print(f"\n  Injury history:\n{day_groups_injury.value_counts().to_string()}")
print(f"\n  Data density:\n{day_groups_density.value_counts().to_string()}")

In [ ]:
# --- Week approach: create proxy groups ---
# Same rationale as day approach: volume and data_density computed from test
# set (no target leakage); injury_history from test set for post-hoc grouping.
week_groups_volume = create_athlete_groups(
    week_test, method="volume", feature_col="week_0_total_kms",
)
week_groups_injury = create_athlete_groups(
    week_test, method="injury_history",
)
week_groups_density = create_athlete_groups(
    week_test, method="data_density",
)

print("Week approach — proxy group distributions:")
print(f"\n  Volume:\n{week_groups_volume.value_counts().to_string()}")
print(f"\n  Injury history:\n{week_groups_injury.value_counts().to_string()}")
print(f"\n  Data density:\n{week_groups_density.value_counts().to_string()}")

---

## 3. Volume-Based Fairness Analysis

Athletes are split by **median total km** on the most recent day/week.
We compare recall, precision, F1, FPR, and AUC-ROC across high- and
low-volume groups. Reference group: **low_volume** (baseline).

### Day approach

In [ ]:
# Per-group metrics — volume, day
metrics_vol_day = compute_group_metrics(
    y_test_day.values, y_pred_day, y_prob_day, day_groups_volume
)
print("Volume groups — Day approach:")
display(metrics_vol_day)

disp_vol_day = compute_disparity_ratios(metrics_vol_day, reference_group="low_volume")
print("\nDisparity ratios (ref = low_volume):")
display(disp_vol_day[["group"] + [c for c in disp_vol_day.columns if "_ratio" in c]])

In [ ]:
fig = plot_group_metrics_bars(
    metrics_vol_day,
    title="Per-Group Metrics — Volume (Day)",
    save_path=Path("fairness/07_group_metrics_volume_day"),
)
plt.show()
plt.close(fig)

fig = plot_disparity_ratios(
    disp_vol_day,
    title="Disparity Ratios — Volume (Day)",
    save_path=Path("fairness/07_disparity_ratios_volume_day"),
)
plt.show()
plt.close(fig)

### Week approach

In [ ]:
# Per-group metrics — volume, week
metrics_vol_week = compute_group_metrics(
    y_test_week.values, y_pred_week, y_prob_week, week_groups_volume
)
print("Volume groups — Week approach:")
display(metrics_vol_week)

disp_vol_week = compute_disparity_ratios(
    metrics_vol_week, reference_group="low_volume"
)
print("\nDisparity ratios (ref = low_volume):")
display(disp_vol_week[["group"] + [c for c in disp_vol_week.columns if "_ratio" in c]])

In [ ]:
fig = plot_group_metrics_bars(
    metrics_vol_week,
    title="Per-Group Metrics — Volume (Week)",
    save_path=Path("fairness/07_group_metrics_volume_week"),
)
plt.show()
plt.close(fig)

fig = plot_disparity_ratios(
    disp_vol_week,
    title="Disparity Ratios — Volume (Week)",
    save_path=Path("fairness/07_disparity_ratios_volume_week"),
)
plt.show()
plt.close(fig)

---

## 4. Injury-History Fairness Analysis

Athletes are split by whether they had **any injury in the training set**.
This is particularly relevant because athletes with prior injuries may have
different training load patterns, and the model may learn to rely on these
patterns differently. Reference group: **never_injured** (baseline).

### Day approach

In [ ]:
# Per-group metrics — injury history, day
metrics_inj_day = compute_group_metrics(
    y_test_day.values, y_pred_day, y_prob_day, day_groups_injury
)
print("Injury history groups — Day approach:")
display(metrics_inj_day)

disp_inj_day = compute_disparity_ratios(
    metrics_inj_day, reference_group="never_injured"
)
print("\nDisparity ratios (ref = never_injured):")
display(disp_inj_day[["group"] + [c for c in disp_inj_day.columns if "_ratio" in c]])

In [ ]:
fig = plot_group_metrics_bars(
    metrics_inj_day,
    title="Per-Group Metrics — Injury History (Day)",
    save_path=Path("fairness/07_group_metrics_injury_history_day"),
)
plt.show()
plt.close(fig)

fig = plot_disparity_ratios(
    disp_inj_day,
    title="Disparity Ratios — Injury History (Day)",
    save_path=Path("fairness/07_disparity_ratios_injury_history_day"),
)
plt.show()
plt.close(fig)

### Week approach

In [ ]:
# Per-group metrics — injury history, week
metrics_inj_week = compute_group_metrics(
    y_test_week.values, y_pred_week, y_prob_week, week_groups_injury
)
print("Injury history groups — Week approach:")
display(metrics_inj_week)

disp_inj_week = compute_disparity_ratios(
    metrics_inj_week, reference_group="never_injured"
)
print("\nDisparity ratios (ref = never_injured):")
display(
    disp_inj_week[["group"] + [c for c in disp_inj_week.columns if "_ratio" in c]]
)

In [ ]:
fig = plot_group_metrics_bars(
    metrics_inj_week,
    title="Per-Group Metrics — Injury History (Week)",
    save_path=Path("fairness/07_group_metrics_injury_history_week"),
)
plt.show()
plt.close(fig)

fig = plot_disparity_ratios(
    disp_inj_week,
    title="Disparity Ratios — Injury History (Week)",
    save_path=Path("fairness/07_disparity_ratios_injury_history_week"),
)
plt.show()
plt.close(fig)

---

## 5. Data-Density Fairness Analysis

Athletes are split by **median observation count** (how many rows each athlete
has in the dataset). Athletes with fewer observations may receive less reliable
predictions because the model has seen fewer examples of their training
patterns. Reference group: **low_density** (baseline).

### Day approach

In [ ]:
# Per-group metrics — data density, day
metrics_den_day = compute_group_metrics(
    y_test_day.values, y_pred_day, y_prob_day, day_groups_density
)
print("Data density groups — Day approach:")
display(metrics_den_day)

disp_den_day = compute_disparity_ratios(
    metrics_den_day, reference_group="low_density"
)
print("\nDisparity ratios (ref = low_density):")
display(disp_den_day[["group"] + [c for c in disp_den_day.columns if "_ratio" in c]])

In [ ]:
fig = plot_group_metrics_bars(
    metrics_den_day,
    title="Per-Group Metrics — Data Density (Day)",
    save_path=Path("fairness/07_group_metrics_data_density_day"),
)
plt.show()
plt.close(fig)

fig = plot_disparity_ratios(
    disp_den_day,
    title="Disparity Ratios — Data Density (Day)",
    save_path=Path("fairness/07_disparity_ratios_data_density_day"),
)
plt.show()
plt.close(fig)

### Week approach

In [ ]:
# Per-group metrics — data density, week
metrics_den_week = compute_group_metrics(
    y_test_week.values, y_pred_week, y_prob_week, week_groups_density
)
print("Data density groups — Week approach:")
display(metrics_den_week)

disp_den_week = compute_disparity_ratios(
    metrics_den_week, reference_group="low_density"
)
print("\nDisparity ratios (ref = low_density):")
display(
    disp_den_week[["group"] + [c for c in disp_den_week.columns if "_ratio" in c]]
)

In [ ]:
fig = plot_group_metrics_bars(
    metrics_den_week,
    title="Per-Group Metrics — Data Density (Week)",
    save_path=Path("fairness/07_group_metrics_data_density_week"),
)
plt.show()
plt.close(fig)

fig = plot_disparity_ratios(
    disp_den_week,
    title="Disparity Ratios — Data Density (Week)",
    save_path=Path("fairness/07_disparity_ratios_data_density_week"),
)
plt.show()
plt.close(fig)

---

## 6. Cross-Method Summary

The heatmaps below consolidate metrics from all three grouping methods into
a single view per approach. This makes it easy to spot which group–method
combination shows the largest performance gap.

In [ ]:
# Summary heatmap — Day approach
all_metrics_day = {
    "volume": metrics_vol_day,
    "injury_history": metrics_inj_day,
    "data_density": metrics_den_day,
}

fig = plot_fairness_summary_heatmap(
    all_metrics_day,
    title="Fairness Summary — Day Approach",
    save_path=Path("fairness/07_summary_heatmap_day"),
)
plt.show()
plt.close(fig)

In [ ]:
# Summary heatmap — Week approach
all_metrics_week = {
    "volume": metrics_vol_week,
    "injury_history": metrics_inj_week,
    "data_density": metrics_den_week,
}

fig = plot_fairness_summary_heatmap(
    all_metrics_week,
    title="Fairness Summary — Week Approach",
    save_path=Path("fairness/07_summary_heatmap_week"),
)
plt.show()
plt.close(fig)

---

## 7. Day vs Week Comparison

We compare disparity patterns across the two temporal approaches to check
whether fairness gaps are consistent or approach-specific.

In [ ]:
# Collect AUC-ROC disparity ratios for all methods, both approaches
comparison_rows = []
for method, disp_day, disp_week in [
    ("volume", disp_vol_day, disp_vol_week),
    ("injury_history", disp_inj_day, disp_inj_week),
    ("data_density", disp_den_day, disp_den_week),
]:
    for _, row in disp_day.iterrows():
        if "auc_roc_ratio" in row.index and pd.notna(row.get("auc_roc_ratio")):
            comparison_rows.append({
                "method": method,
                "group": row["group"],
                "approach": "day",
                "auc_roc_ratio": row["auc_roc_ratio"],
            })
    for _, row in disp_week.iterrows():
        if "auc_roc_ratio" in row.index and pd.notna(row.get("auc_roc_ratio")):
            comparison_rows.append({
                "method": method,
                "group": row["group"],
                "approach": "week",
                "auc_roc_ratio": row["auc_roc_ratio"],
            })

cmp_df = pd.DataFrame(comparison_rows)
# Exclude reference rows (ratio ≈ 1.0)
cmp_df = cmp_df[~np.isclose(cmp_df["auc_roc_ratio"], 1.0)].copy()

if not cmp_df.empty:
    cmp_df["label"] = cmp_df["method"] + " | " + cmp_df["group"]
    labels = cmp_df["label"].unique()
    x = np.arange(len(labels))
    width = 0.35

    fig, ax = plt.subplots(figsize=(12, max(4, len(labels) * 0.8)))
    day_vals = []
    week_vals = []
    for label in labels:
        d = cmp_df[(cmp_df["label"] == label) & (cmp_df["approach"] == "day")]
        day_vals.append(d["auc_roc_ratio"].iloc[0] if len(d) > 0 else 0)
        w = cmp_df[(cmp_df["label"] == label) & (cmp_df["approach"] == "week")]
        week_vals.append(w["auc_roc_ratio"].iloc[0] if len(w) > 0 else 0)

    ax.barh(x + width / 2, day_vals, width, label="Day", color="#2196F3", alpha=0.85)
    ax.barh(x - width / 2, week_vals, width, label="Week", color="#FF9800", alpha=0.85)
    ax.axvline(1.0, color="#9E9E9E", linestyle="--", linewidth=1.5, label="Parity")
    ax.set_yticks(x)
    ax.set_yticklabels(labels)
    ax.set_xlabel("AUC-ROC Disparity Ratio")
    ax.set_title("Day vs Week — AUC-ROC Disparity Ratios", fontweight="bold")
    ax.legend()
    ax.invert_yaxis()
    fig.tight_layout()

    save_figure(
        fig, "07_day_vs_week_disparity_comparison",
        subdir="fairness", close=False,
    )
    plt.show()
    plt.close(fig)
else:
    print("No disparity ratios available for comparison.")

---

## 8. Critical Discussion: Limits Without Demographics

### Why proxy groups?

The Lovdal et al. (2021) dataset contains **no demographic attributes** —
athlete IDs are masked, and age, sex, ethnicity, and other protected
characteristics are unavailable. Without these attributes, standard fairness
metrics (demographic parity, equalized odds, calibration across groups) cannot
be computed directly.

Proxy groups are a **pragmatic alternative**: they reveal performance gaps
across observable sub-populations that may correlate with unobserved protected
attributes. For example, training volume differences may partly reflect
age- or experience-related patterns.

### What proxy groups can reveal

- **Performance gaps**: if the model achieves AUC-ROC = 0.75 for high-volume
  athletes but only 0.55 for low-volume athletes, there is a systematic gap
  regardless of the underlying cause.
- **Recall disparities**: if the model misses more injuries in one group,
  athletes in that group face higher undetected risk — a safety concern.
- **FPR differences**: higher false positive rates in one group mean more
  unnecessary training restrictions for those athletes.

### What proxy groups cannot reveal

- **Demographic bias**: a proxy group is not a protected attribute. Equal
  performance across volume groups does not guarantee equal performance
  across sex or age groups.
- **Intersectional effects**: proxy groups are univariate splits. Real-world
  bias is often intersectional (e.g., young + high-volume + female).
- **Causal attribution**: a disparity in model performance may reflect
  data collection patterns, class imbalance within groups, or genuine
  differences in injury predictability — not necessarily model bias.

### The four-fifths rule (adapted)

The US Equal Employment Opportunity Commission (EEOC) four-fifths rule states
that a selection rate below 80% of the highest group's rate constitutes
evidence of adverse impact. Adapted to ML metrics:

- A **recall ratio below 0.8** means the model detects injuries at less than
  80% of the reference group's rate — a potential safety concern.
- An **AUC-ROC ratio below 0.8** indicates substantially worse overall
  discrimination for one group.

These thresholds are **guidelines, not absolutes**. Small groups, class
imbalance, and proxy-group limitations all affect interpretation.

### Recommendations

1. **If demographics become available**: compute standard fairness metrics
   (equalized odds, calibration, predictive parity) across true protected
   groups.
2. **Monitor in deployment**: track per-athlete prediction accuracy over time
   to detect emerging performance gaps.
3. **Report transparently**: acknowledge proxy-group limitations in any
   deployment documentation. Proxy fairness is better than no fairness
   analysis, but it is not a substitute for demographic auditing.

---

## 9. Summary

### Pipeline flow

```
data/processed/day_best_model.pkl + week_best_model.pkl
  → load_model()
data/processed/day_test.parquet + week_test.parquet
  → load_splits()
  → create_athlete_groups() [volume, injury_history, data_density]
  → compute_group_metrics() [per-group recall, precision, F1, FPR, AUC-ROC]
  → compute_disparity_ratios() [relative to reference group]
  → Visualization: bar charts, disparity plots, summary heatmaps
  → Cross-comparison: day vs week disparity ratios
  → All figures saved to reports/figures/fairness/
```

### Figures generated

All figures saved to `reports/figures/fairness/`:
- 6 per-group metric bar charts (3 methods × 2 approaches)
- 6 disparity ratio charts (3 methods × 2 approaches)
- 2 summary heatmaps (day + week)
- 1 day-vs-week disparity comparison

**Total: 15 figures**

### Key findings

1. **Proxy groups** provide a pragmatic fairness assessment in the absence
   of demographic data — three grouping strategies offer complementary views.
2. **Per-group metrics** reveal whether the model performs equitably across
   athlete sub-populations defined by training volume, injury history, and
   data availability.
3. **Disparity ratios** quantify the magnitude of any performance gaps,
   with the four-fifths rule as an interpretive guideline.
4. **Day vs week comparison** shows whether fairness patterns are consistent
   across temporal granularities or approach-specific.
5. **Limitations acknowledged**: proxy groups cannot replace demographic
   auditing, and observed disparities may reflect data characteristics
   rather than model bias.

### Next steps

- **Notebook 08**: Comparison summary — day vs week side-by-side, paper
  benchmark comparison, final recommendation